단순확율대치법
단순확률대치법(Hot-deck random / Random imputation)**은

| 평균대치   | 단순확률대치       |
| ------ | ------------ |
| 평균값 넣음 | 실제 관측값 중 랜덤  |
| 분산 감소  | 분산 유지(상대적으로) |
| 왜곡 심함  | 조금 더 자연스러움   |

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({"score": [80, 90, 70, np.nan, 85]})

observed = df["score"].dropna()

df["score"] = df["score"].apply(
    lambda x: np.random.choice(observed) if pd.isna(x) else x
)

print(df)

3️⃣ 알고리즘 단계

1️⃣ 결측 없는 변수들 기준으로 거리 계산
2️⃣ 가장 가까운 k개 이웃 선택
3️⃣ 평균(연속형) 또는 최빈값(범주형)으로 대치

In [ ]:
import numpy as np
from sklearn.impute import KNNImputer

data = np.array([
    [170, 65],
    [168, 60],
    [172, 70],
    [171, np.nan]
])

imputer = KNNImputer(n_neighbors=2)
filled = imputer.fit_transform(data)

print(filled)

2️⃣ 전체 절차 (시험에서 가장 중요)
🔹 Step 1️⃣ 대치(Imputation) – m개의 데이터 생성

결측값을 예측 모델로 추정

랜덤 오차를 포함해서 대치

이 과정을 m번 반복

예: m=5라면
→ 5개의 완성 데이터셋 생성

🔹 Step 2️⃣ 각 데이터셋에서 분석

각 데이터셋에서

회귀분석

평균 비교

분산분석

등 수행

→ 추정치 θ₁, θ₂, …, θₘ 얻음

🔹 Step 3️⃣ 결과 통합 (Rubin’s Rule)
최종 추정치

𝜃ˉ=1/𝑚*∑theta𝑖

최종 분산

𝑇=𝑊+(1+1/𝑚)𝐵

W: 평균 내부 분산
B: 대치 간 분산
👉 불확실성까지 반영됨

1. 다중대치가 뭐냐 (핵심 정의)

결측값을 한 번만 채우는 게 아니라, 여러 번(m번) 채워서 여러 개의 완전한 데이터셋을 만든 다음 결과를 합치는 방법


2. 왜 쓰냐 (단일 대치의 문제)

예를 들어:

NaN → 평균으로 채움

이건 문제 있음:

불확실성 무시됨
분산이 줄어듦 (데이터가 과도하게 “깔끔”해짐)
모델이 과신(overconfidence)

👉 즉, 하나의 값으로 확정해버리는 게 문제

3. 다중대치 핵심 아이디어

결측값을 이렇게 본다:

"정답이 하나가 아니라, 여러 가능성이 있다"

그래서:

→ 여러 번 랜덤하게 채움
→ 여러 개 데이터셋 생성
→ 각각 분석
→ 결과 평균냄

In [ ]:
# sklearn (유사 다중대치)

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import numpy as np

data = np.array([
    [170, np.nan, 30],
    [165, 55, 28],
    [172, 70, 35]
])

imputer = IterativeImputer(max_iter=10, random_state=0)
filled = imputer.fit_transform(data)
print(filled)

# 진짜 다중대치 (miceforest)
import miceforest as mf

kernel = mf.ImputationKernel(
    data,
    datasets=5,   # m=5
    save_all_iterations=True
)

kernel.mice(5)

completed_data = kernel.complete_data(dataset=0)